In [1]:
# ============================================================================
# PhiSat-2 Session 2929 -- FULL CORRECTED DIAGNOSTIC (v3: rigorous footprint)
# ============================================================================

import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.control import GroundControlPoint
from rasterio.transform import from_gcps
from shapely.geometry import box, Polygon
from shapely.ops import unary_union
import matplotlib.pyplot as plt

PROJECT   = Path("/root/projects/iride_onboard-burnscar-mapper")
PHI_DIR   = PROJECT / "data" / "Phi_dataset"
RR_ZIP    = PROJECT / "data" / "RR_2929.zip"
EFFIS_ZIP = PROJECT / "data" / "PHISAT_83757be9667e4cb4bb9be256a8770f9b.zip"
OUT_DIR   = PROJECT / "cgi_feedback"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SESSION_ID = 2929
MAX_DAYS_AFTER_FIRE = 90

def vsizip(zip_path, inner):
    return f"/vsizip/{Path(zip_path).resolve()}/{inner}"

def stretch(a, mask, lo=2, hi=98):
    p1, p2 = np.percentile(a[mask], [lo, hi])
    return np.clip((a - p1) / (p2 - p1 + 1e-9), 0, 1)

bc_matches = [d for d in PHI_DIR.glob(f"PHISAT-2_L1_000{SESSION_ID:06d}_*") if d.is_dir()]
assert len(bc_matches) == 1, f"expected exactly 1 BC folder match, got {bc_matches}"
BC_DIR = bc_matches[0]

# ============================================================================
# STEP 0 -- CONFIRMED band order (3 independent sources agree: wavelength
# metadata, CGI's own quicklook R=idx3/G=idx2/B=idx1 logic, visual check)
# ============================================================================
IDX = dict(BLUE=1, GREEN=2, RED=3, RE1=4, RE2=5, RE3=6, NIR=7)  # idx0=625nm, unlabeled
print("="*90)
print("STEP 0 -- Band order (CONFIRMED via 3 independent sources)")
print("="*90)
print(f"Using: {IDX}")

mb_path = vsizip(RR_ZIP, "RR/scene_0_RR_multiband.tiff")
with rasterio.open(mb_path) as src:
    stack = src.read().astype(np.float32)
    H, W = src.height, src.width
    rr_native_crs = src.crs

valid = np.ones((H, W), bool)
for b in range(8):
    valid &= (stack[b] > 0)
print(f"stack shape: {stack.shape}, valid pixels: {valid.sum():,}/{H*W:,} "
      f"({100*valid.sum()/(H*W):.1f}%)")

# ============================================================================
# STEP 1 -- GEOLOCATION: load grid, fit transform, check residuals FOR THIS
# SESSION specifically (not reused from a different one)
# ============================================================================
print("\n" + "="*90)
print("STEP 1 -- GEOLOCATION")
print("="*90)

gl = json.loads((BC_DIR / "geolocation" / "GL_scene_0.json").read_text())
pts = gl["Geolocated_Points"]
n = int(np.sqrt(len(pts)))
assert n * n == len(pts), f"not a perfect square: {len(pts)}"

lons = np.array([p["Lon"] for p in pts])
lats = np.array([p["Lat"] for p in pts])
xs   = np.array([p["X_coordinate"] for p in pts])
ys   = np.array([p["Y_coordinate"] for p in pts])

gcps = [GroundControlPoint(row=p["Y_coordinate"], col=p["X_coordinate"],
                           x=p["Lon"], y=p["Lat"]) for p in pts]
fitted_transform = from_gcps(gcps)
PHISAT_CRS = "EPSG:4326"

residuals = []
for p in pts:
    pred_lon, pred_lat = fitted_transform * (p["X_coordinate"], p["Y_coordinate"])
    residuals.append(np.hypot(pred_lon - p["Lon"], pred_lat - p["Lat"]))
residuals = np.array(residuals) * 111000

print(f"grid: {len(pts)} points ({n}x{n}), THIS session's own geolocation file")
print(f"fit residual: mean={residuals.mean():.1f}m  max={residuals.max():.1f}m")
print(f"** IMPORTANT: this measures whether the fitted transform reproduces")
print(f"** the grid's OWN points -- it does NOT prove the grid itself is")
print(f"** correct against ground truth. It could be self-consistent and")
print(f"** still shifted by a constant amount in the real world.")

# Residual spatial pattern -- smooth structure (not random noise) would
# indicate genuine non-affine distortion, not just point-level jitter
fig, ax = plt.subplots(figsize=(7, 7))
sc = ax.scatter(xs, ys, c=residuals, cmap="viridis", s=10)
ax.invert_yaxis()
plt.colorbar(sc, label="residual (m)")
ax.set_title(f"Session {SESSION_ID}: GCP fit residual pattern")
fig.savefig(OUT_DIR / "step1_residual_pattern.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {OUT_DIR / 'step1_residual_pattern.png'}")

# Visual check: raw grid points at stated pixel positions, coloured by
# latitude -- independent of the fitted transform entirely
cir = np.dstack([stretch(stack[IDX["NIR"]], valid), stretch(stack[IDX["RED"]], valid),
                 stretch(stack[IDX["GREEN"]], valid)])

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(cir, interpolation="nearest")
sc = ax.scatter(xs, ys, c=lats, cmap="coolwarm", s=8, edgecolors="none")
plt.colorbar(sc, label="latitude")
ax.set_title(f"Session {SESSION_ID}: raw geolocation grid vs image\n"
            f"(low-latitude points should sit on visible sea)")
fig.savefig(OUT_DIR / "step1_geoloc_visual_check.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {OUT_DIR / 'step1_geoloc_visual_check.png'}")
print(">>> RECOMMENDED: pick one distinctive landmark visible in this image")
print(">>> (a small bay, reservoir, harbour) and manually check its computed")
print(">>> lon/lat against Google Maps. This is the one check that validates")
print(">>> against TRUE ground truth rather than internal self-consistency.")

corners_geo = [fitted_transform * c for c in [(0,0),(W,0),(0,H),(W,H)]]
ext_geo = (min(c[0] for c in corners_geo), max(c[0] for c in corners_geo),
          min(c[1] for c in corners_geo), max(c[1] for c in corners_geo))

# ============================================================================
# STEP 2 -- CRS MATCH: confirm PhiSat-2 and EFFIS are reconciled to the same
# projection before doing any spatial operation between them
# ============================================================================
print("\n" + "="*90)
print("STEP 2 -- CRS / PROJECTION MATCH")
print("="*90)
print(f"RR raster's own embedded CRS: {rr_native_crs}")
print(f"Geolocation grid CRS (used for fitted_transform): {PHISAT_CRS}")
assert str(rr_native_crs).upper() == PHISAT_CRS.upper(), \
    f"MISMATCH: raster embedded CRS {rr_native_crs} != {PHISAT_CRS}"
print("MATCH: raster's embedded CRS agrees with the geolocation grid CRS")

with zipfile.ZipFile(EFFIS_ZIP) as zf:
    names = [nm for nm in zf.namelist() if nm.lower().endswith((".geojson", ".json"))]
gdf_raw = gpd.read_file(f"zip://{EFFIS_ZIP}!{names[0]}")
print(f"EFFIS native CRS (as delivered): {gdf_raw.crs}")

gdf = gdf_raw[gdf_raw.geometry.notna() & ~gdf_raw.geometry.is_empty].copy()
gdf["finaldate"] = pd.to_datetime(gdf["finaldate"], utc=True)
gdf_filtered = gdf.to_crs(PHISAT_CRS)
gdf_filtered = gdf_filtered[gdf_filtered["area_ha"] >= 10].copy()
assert str(gdf_filtered.crs).upper() == PHISAT_CRS.upper()
print(f"EFFIS reprojected {gdf_raw.crs} -> {gdf_filtered.crs}: CONFIRMED MATCH")

# ============================================================================
# STEP 3 -- RIGOROUS FOOTPRINT: reconstruct the exact scanned area as a mesh
# of real quadrilaterals from the 129x129 grid. No heuristic buffer size, no
# convexity assumption -- this IS the sensor's true footprint, built directly
# from verified geolocation points. Supersedes both earlier broken attempts:
#   v1 (perimeter-only polygon) -- assumed convex, failed on this coastline
#   v2 (point-buffer union)     -- heuristic, parameter-dependent, untested
#   v3 (this)                   -- exact reconstruction, no free parameters
# ============================================================================
print("\n" + "="*90)
print("STEP 3 -- RIGOROUS FOOTPRINT (quad-mesh reconstruction)")
print("="*90)

lon_grid = lons.reshape(n, n)
lat_grid = lats.reshape(n, n)

quads = []
for i in range(n - 1):
    for j in range(n - 1):
        q = Polygon([
            (lon_grid[i, j],     lat_grid[i, j]),
            (lon_grid[i, j+1],   lat_grid[i, j+1]),
            (lon_grid[i+1, j+1], lat_grid[i+1, j+1]),
            (lon_grid[i+1, j],   lat_grid[i+1, j]),
        ])
        if q.is_valid and q.area > 0:
            quads.append(q)

print(f"building footprint from {len(quads)} grid cells (this can take ~10-30s)...")
true_footprint = unary_union(quads)
rect_bbox = box(lons.min(), lats.min(), lons.max(), lats.max())

print(f"quad-mesh footprint area: {true_footprint.area:.5f} sq.deg")
print(f"rectangular bbox area:    {rect_bbox.area:.5f} sq.deg "
      f"({100*(rect_bbox.area/true_footprint.area - 1):.1f}% larger)")

# ============================================================================
# STEP 4 -- FIRE MATCHING against the rigorous footprint, with full audit
# trail comparing all methods tried so far for transparency
# ============================================================================
print("\n" + "="*90)
print("STEP 4 -- FIRE MATCHING (audit trail across all attempted methods)")
print("="*90)

m = re.search(r"_(\d{14})_", BC_DIR.name)
acq_start = pd.to_datetime(m.group(1), format="%Y%m%d%H%M%S", utc=True)
print(f"acquisition date: {acq_start}")

n_rect = len(gdf_filtered[gdf_filtered.intersects(rect_bbox)])
n_mesh = len(gdf_filtered[gdf_filtered.intersects(true_footprint)])
print(f"  v0 (rectangle, too permissive):  {n_rect} fires overlap")
print(f"  v3 (quad-mesh, rigorous):        {n_mesh} fires overlap")

overlapping = gdf_filtered[gdf_filtered.intersects(true_footprint)].copy()
overlapping["days_after_fire"] = (acq_start - overlapping["finaldate"]).dt.days
fires_geom = overlapping[
    (overlapping["days_after_fire"] >= 0) &
    (overlapping["days_after_fire"] <= MAX_DAYS_AFTER_FIRE)
]
print(f"  passing recency filter (<= {MAX_DAYS_AFTER_FIRE} days): {len(fires_geom)}")

# ============================================================================
# STEP 5 -- PLOT: fires over the true footprint, using the rigorous mesh
# ============================================================================
g_corrected = fires_geom.to_crs(PHISAT_CRS)
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(cir, extent=ext_geo, origin="upper", interpolation="nearest")
if len(g_corrected):
    g_corrected.boundary.plot(ax=ax, color="cyan", linewidth=1.3)
ax.set_xlim(ext_geo[0], ext_geo[1]); ax.set_ylim(ext_geo[2], ext_geo[3])
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title(f"Session {SESSION_ID}: {len(g_corrected)} EFFIS fires\n"
            f"(quad-mesh footprint -- rigorous, no heuristic parameters)")
fig.savefig(OUT_DIR / "step5_fires_rigorous_footprint.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"\nSaved: {OUT_DIR / 'step5_fires_rigorous_footprint.png'}")
print(">>> OPEN AND CONFIRM: every polygon should sit on visible land now.")

# ============================================================================
# STEP 6 -- RR VALUES: per-band stats + water spectral-shape check.
# Real open water should show Blue > Green > Red > NIR (~0). Seeing this
# order INVERTED is a sharp, physically clean diagnostic independent of any
# assumption about scale factor.
# ============================================================================
print("\n" + "="*90)
print("STEP 6 -- RR VALUES: per-band stats + water spectral-shape check")
print("="*90)

# Confirm these boxes actually look like water/land in step5's image before
# trusting this -- adjust if not
water_box = (slice(3600, 3950), slice(300, 3900))
land_box  = (slice(100, 700), slice(1200, 2600))
water_valid_px = valid[water_box]
land_valid_px  = valid[land_box]

print(f"{'role':<8}{'water mean':>12}{'land mean':>12}")
water_means, land_means = {}, {}
for role, idx in IDX.items():
    wm = stack[idx][water_box][water_valid_px].mean()
    lm = stack[idx][land_box][land_valid_px].mean()
    water_means[role] = wm; land_means[role] = lm
    print(f"{role:<8}{wm:>12.2f}{lm:>12.2f}")

print(f"\nExpected water spectral order: Blue > Green > Red > NIR (~0)")
print(f"Observed water order (high to low): "
      f"{sorted(['BLUE','GREEN','RED','NIR'], key=lambda r: -water_means[r])}")
if water_means["RED"] > water_means["GREEN"] > water_means["BLUE"]:
    print(">>> INVERTED relative to expected clear-water behaviour.")
    print(">>> Worth confirming visually that this sample box is genuinely")
    print(">>> open water (not shallow/turbid coastal water or partial land).")

# ============================================================================
# STEP 7 -- NDVI: global AND land-only (land-only is the number that matters,
# global is diluted by water which is SUPPOSED to be strongly negative)
# ============================================================================
print("\n" + "="*90)
print("STEP 7 -- NDVI (global vs land-only)")
print("="*90)

ndvi_full = (stack[IDX["NIR"]] - stack[IDX["RED"]]) / (stack[IDX["NIR"]] + stack[IDX["RED"]] + 1e-6)
ndwi_full = (stack[IDX["GREEN"]] - stack[IDX["NIR"]]) / (stack[IDX["GREEN"]] + stack[IDX["NIR"]] + 1e-6)

water_est = valid & (ndwi_full > 0.0)
land_est  = valid & (ndwi_full <= 0.0)

print(f"estimated water: {water_est.sum():,} ({100*water_est.sum()/valid.sum():.1f}%)")
print(f"estimated land:  {land_est.sum():,} ({100*land_est.sum()/valid.sum():.1f}%)")
print(f"(NDWI-based split, an approximation -- not a perfect classifier)")
print(f"\nNDVI global mean:    {ndvi_full[valid].mean():.3f}")
print(f"NDVI on water:       {ndvi_full[water_est].mean():.3f}  (expected strongly negative -- normal)")
print(f"NDVI on LAND only:   {ndvi_full[land_est].mean():.3f}   "
      f"positive fraction={100*(ndvi_full[land_est]>0).sum()/land_est.sum():.2f}%")
print(f">>> This land-only number is what actually matters, not the global one.")

fig, ax = plt.subplots(figsize=(9, 9))
im = ax.imshow(np.where(valid, ndvi_full, np.nan), extent=ext_geo, origin="upper",
               cmap="RdYlGn", vmin=-0.7, vmax=0.3)
ax.set_xlim(ext_geo[0], ext_geo[1]); ax.set_ylim(ext_geo[2], ext_geo[3])
plt.colorbar(im, label="NDVI")
ax.set_title(f"Session {SESSION_ID}: NDVI map")
fig.savefig(OUT_DIR / "step7_ndvi_map.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {OUT_DIR / 'step7_ndvi_map.png'}")

# ============================================================================
# STEP 8 -- PER-FIRE local comparison (buffered ring, not whole-scene) using
# the CORRECTED fire set from the rigorous footprint
# ============================================================================
print("\n" + "="*90)
print("STEP 8 -- PER-FIRE local burnt-vs-surrounding comparison")
print("="*90)

BUFFER_DEG = 0.003
local_results = []
for _, fire in fires_geom.iterrows():
    fire_native = gpd.GeoSeries([fire.geometry], crs=fires_geom.crs).to_crs(PHISAT_CRS).iloc[0]
    ring = fire_native.buffer(BUFFER_DEG).difference(fire_native)

    from rasterio.features import rasterize
    fire_mask = rasterize([(fire_native, 1)], out_shape=(H, W),
                          transform=fitted_transform, fill=0, dtype=np.uint8)
    ring_mask = rasterize([(ring, 1)], out_shape=(H, W),
                         transform=fitted_transform, fill=0, dtype=np.uint8)

    ins = (fire_mask == 1) & valid
    local_out = (ring_mask == 1) & valid & ~ins
    if ins.sum() < 30 or local_out.sum() < 30:
        continue

    ndvi_in, ndvi_out = ndvi_full[ins].mean(), ndvi_full[local_out].mean()
    local_results.append(dict(
        id=fire["id"], area_ha=fire["area_ha"],
        agriculture_pct=fire["agriculture_percent"],
        sclerophyllous_pct=fire["sclerophillous_vegetation_percent"],
        px_count=int(ins.sum()), ndvi_in=round(ndvi_in, 3),
        ndvi_out=round(ndvi_out, 3), ndvi_diff=round(ndvi_in - ndvi_out, 3),
    ))

local_df = pd.DataFrame(local_results).sort_values("area_ha", ascending=False)
print(f"{len(local_df)} fires with sufficient local comparison data")
display(local_df)

# ============================================================================
# STEP 9 -- SUMMARY + QUESTIONS FOR CGI
# ============================================================================
print("\n" + "="*90)
print("SUMMARY")
print("="*90)
print(f"""
Session: {SESSION_ID} ({BC_DIR.name})

CONFIRMED (high confidence):
- Band order: idx1=Blue(490), idx2=Green(560), idx3=Red(665), idx4=RE1(705),
  idx5=RE2(740), idx6=RE3(783), idx7=NIR(842). Three independent sources agree.
- CRS: PhiSat-2 native CRS and EFFIS (after reprojection) both EPSG:4326, confirmed.
- Geolocation grid is internally self-consistent (mean {residuals.mean():.1f}m,
  max {residuals.max():.1f}m residual against its own 16,641 points).
- Fire matching now uses a rigorous quad-mesh footprint reconstructed directly
  from the geolocation grid, not a heuristic approximation.

STILL UNVERIFIED / WORTH DOUBTING:
- Residual check above is SELF-consistency only. We have not checked this
  grid against an independent ground-truth landmark. Recommend manually
  checking one identifiable feature (bay, reservoir, harbour) against Google
  Maps before treating geolocation as fully validated.
- NDWI-based land/water split is an approximation, not a validated classifier.
- EFFIS `finaldate` reliability as "fire fully out" is assumed, not verified.

QUESTIONS FOR CGI:
1. Please confirm officially: band order is
   [625nm(unlabeled), 490nm-Blue, 560nm-Green, 665nm-Red, 705nm-RE1,
   740nm-RE2, 783nm-RE3, 842nm-NIR] -- we've inferred this from your metadata
   and quicklook generation logic, but haven't had it confirmed directly.
2. What is the intended physical unit and scale factor for RR pixel values?
   (per-band max ranges ~44-206 on a uint16 container -- consistent with
   percent reflectance directly, but not confirmed.)
3. A sample of open water shows reflectance ordering Red > Green > Blue,
   the opposite of typical clear-water spectral behaviour (normally
   Blue > Green > Red > NIR~0). Is there a known band-dependent gain
   difference in the visible bands we should account for?
4. Land pixels show mean NDVI of {ndvi_full[land_est].mean():.3f} with only
   {100*(ndvi_full[land_est]>0).sum()/land_est.sum():.1f}% positive, well
   below typical values for Mediterranean summer vegetation. Could there be
   an uncorrected offset specifically in the red-edge/NIR bands relative to
   red?
5. Should future RR deliveries include their own geolocation grid, or is
   reusing the BC-stage grid for the same session (since AC/RR are purely
   radiometric steps) the correct approach going forward?
""")
print("="*90)
print(f"Images saved to: {OUT_DIR}")
print("Open step5 first (fire overlay -- should be clean now), then step7 (NDVI map).")
print("="*90)

STEP 0 -- Band order (CONFIRMED via 3 independent sources)
Using: {'BLUE': 1, 'GREEN': 2, 'RED': 3, 'RE1': 4, 'RE2': 5, 'RE3': 6, 'NIR': 7}
stack shape: (8, 4096, 4096), valid pixels: 16,776,321/16,777,216 (100.0%)

STEP 1 -- GEOLOCATION
grid: 16641 points (129x129), THIS session's own geolocation file
fit residual: mean=6.2m  max=24.5m
** IMPORTANT: this measures whether the fitted transform reproduces
** the grid's OWN points -- it does NOT prove the grid itself is
** correct against ground truth. It could be self-consistent and
** still shifted by a constant amount in the real world.
Saved: /root/projects/iride_onboard-burnscar-mapper/cgi_feedback/step1_residual_pattern.png
Saved: /root/projects/iride_onboard-burnscar-mapper/cgi_feedback/step1_geoloc_visual_check.png
>>> RECOMMENDED: pick one distinctive landmark visible in this image
>>> (a small bay, reservoir, harbour) and manually check its computed
>>> lon/lat against Google Maps. This is the one check that validates
>>> agains

,id,area_ha,agriculture_pct,sclerophyllous_pct,px_count,ndvi_in,ndvi_out,ndvi_diff
6,281556,1153,58.956522,40.086957,251151,-0.345,-0.341,-0.004
28,281259,616,53.387097,18.387097,242596,-0.369,-0.355,-0.014
3,271542,528,41.541353,24.248120,238302,-0.285,-0.281,-0.004
4,274690,527,24.618321,0.763359,237971,-0.310,-0.310,0.001
5,281555,518,67.906067,11.154599,14144,-0.362,-0.360,-0.002
7,276441,390,36.553525,0.000000,175744,-0.646,-0.632,-0.014
19,271541,315,5.750799,0.958466,142328,-0.279,-0.280,0.001
0,270865,313,44.827586,0.000000,4910,-0.616,-0.613,-0.003
1,271189,158,6.172840,0.000000,71421,-0.627,-0.626,-0.002
27,275741,154,25.333333,42.666667,63931,-0.273,-0.283,0.010



SUMMARY

Session: 2929 (PHISAT-2_L1_000002929_20250730100820_20250730100823_27FAA0CD)

CONFIRMED (high confidence):
- Band order: idx1=Blue(490), idx2=Green(560), idx3=Red(665), idx4=RE1(705),
  idx5=RE2(740), idx6=RE3(783), idx7=NIR(842). Three independent sources agree.
- CRS: PhiSat-2 native CRS and EFFIS (after reprojection) both EPSG:4326, confirmed.
- Geolocation grid is internally self-consistent (mean 6.2m,
  max 24.5m residual against its own 16,641 points).
- Fire matching now uses a rigorous quad-mesh footprint reconstructed directly
  from the geolocation grid, not a heuristic approximation.

STILL UNVERIFIED / WORTH DOUBTING:
- Residual check above is SELF-consistency only. We have not checked this
  grid against an independent ground-truth landmark. Recommend manually
  checking one identifiable feature (bay, reservoir, harbour) against Google
  Maps before treating geolocation as fully validated.
- NDWI-based land/water split is an approximation, not a validated classif

In [3]:
from shapely.geometry import Point

fires_native = fires_geom.to_crs(PHISAT_CRS).copy()
fires_native["orig_area"] = fires_native.geometry.area
fires_native["clipped_geom"] = fires_native.geometry.intersection(true_footprint)
fires_native["clipped_area"] = fires_native["clipped_geom"].area
fires_native["pct_in_scene"] = 100 * fires_native["clipped_area"] / fires_native["orig_area"]

display(fires_native[["id", "area_ha", "pct_in_scene"]].sort_values("pct_in_scene"))

/tmp/ipykernel_3445391/987605612.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  fires_native["orig_area"] = fires_native.geometry.area
/tmp/ipykernel_3445391/987605612.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  fires_native["clipped_area"] = fires_native["clipped_geom"].area


,id,area_ha,pct_in_scene
1747,270865,313,3.199905
1874,281555,518,6.007035
3196,270868,76,10.764386
1875,281556,1153,48.205896
5819,281259,616,87.357425
5799,275741,154,92.098807
4077,271236,10,100.000000
4068,271217,12,100.000000
4216,271541,315,100.000000
3189,270867,22,100.000000


In [4]:
g_clipped = fires_native.set_geometry("clipped_geom")
g_clipped = g_clipped[~g_clipped.geometry.is_empty]

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(cir, extent=ext_geo, origin="upper", interpolation="nearest")
g_clipped.boundary.plot(ax=ax, color="cyan", linewidth=1.3)
ax.set_xlim(ext_geo[0], ext_geo[1]); ax.set_ylim(ext_geo[2], ext_geo[3])
ax.set_title(f"Session {SESSION_ID}: fires CLIPPED to true footprint")
fig.savefig(OUT_DIR / "step5b_clipped_fires.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print("saved")

saved


In [5]:
# Pick a few fires whose clipped geometry still visually sits in open water,
# based on your screenshot -- these look like the isolated round blob
# clusters south/southeast of the visible coastline
suspect_ids = fires_native[fires_native["pct_in_scene"] > 50]["id"].tolist()
print(f"checking {len(suspect_ids)} fire(s) with substantial in-scene area")

for fid in suspect_ids:
    row = gdf_filtered[gdf_filtered["id"] == fid].iloc[0]  # raw EFFIS geometry, WGS84, nothing else touched
    c = row.geometry.centroid
    minx, miny, maxx, maxy = row.geometry.bounds
    print(f"\nfire id {fid}  (area {row['area_ha']:.0f} ha)")
    print(f"  centroid (lon, lat): {c.x:.5f}, {c.y:.5f}")
    print(f"  bbox: ({minx:.5f}, {miny:.5f}) to ({maxx:.5f}, {maxy:.5f})")
    print(f"  https://www.google.com/maps?q={c.y:.5f},{c.x:.5f}")

checking 25 fire(s) with substantial in-scene area

fire id 271189  (area 158 ha)
  centroid (lon, lat): 13.38652, 37.36557
  bbox: (13.37454, 37.35737) to (13.39598, 37.37530)
  https://www.google.com/maps?q=37.36557,13.38652

fire id 271216  (area 77 ha)
  centroid (lon, lat): 13.32334, 37.41548
  bbox: (13.31311, 37.40436) to (13.33109, 37.42307)
  https://www.google.com/maps?q=37.41548,13.32334

fire id 271542  (area 528 ha)
  centroid (lon, lat): 13.42256, 37.45120
  bbox: (13.40089, 37.43964) to (13.44683, 37.46333)
  https://www.google.com/maps?q=37.45120,13.42256

fire id 274690  (area 527 ha)
  centroid (lon, lat): 13.42145, 37.42744
  bbox: (13.39289, 37.41369) to (13.44693, 37.44406)
  https://www.google.com/maps?q=37.42744,13.42145

fire id 276441  (area 390 ha)
  centroid (lon, lat): 13.41399, 37.36295
  bbox: (13.39154, 37.34589) to (13.43313, 37.37731)
  https://www.google.com/maps?q=37.36295,13.41399

fire id 270168  (area 17 ha)
  centroid (lon, lat): 13.29973, 37.3974

In [8]:
test_row, test_col = 200, 500  # adjust to match where the reservoir sits in your image
lon_ours, lat_ours = fitted_transform * (test_col, test_row)
print(f"Our computed lon/lat: {lon_ours:.5f}, {lat_ours:.5f}")
print(f"https://www.google.com/maps?q={lat_ours:.5f},{lon_ours:.5f}")

Our computed lon/lat: 13.48486, 37.46750
https://www.google.com/maps?q=37.46750,13.48486


In [9]:
sample_fire = gdf_filtered[gdf_filtered["id"] == suspect_ids[0]].iloc[0]
geom = sample_fire.geometry

if geom.geom_type == "MultiPolygon":
    first_poly = list(geom.geoms)[0]
else:
    first_poly = geom

print("Raw EFFIS geometry (before ANY reprojection), first coordinate:")
print(list(first_poly.exterior.coords)[0])
print(f"gdf_filtered.crs at this point: {gdf_filtered.crs}")
print(f"geometry type: {geom.geom_type}, number of parts: "
      f"{len(geom.geoms) if geom.geom_type == 'MultiPolygon' else 1}")

Raw EFFIS geometry (before ANY reprojection), first coordinate:
(13.37827069754159, 37.37503643844462)
gdf_filtered.crs at this point: EPSG:4326
geometry type: MultiPolygon, number of parts: 1


In [10]:
# Identify which specific fires have their centroid over water, using OUR
# own NDWI-based water mask, independent of the footprint/clipping logic
suspects = []
for _, fire in fires_native.iterrows():
    c = fire.geometry.centroid
    col, row = ~fitted_transform * (c.x, c.y)
    row, col = int(row), int(col)
    if 0 <= row < H and 0 <= col < W:
        is_water_here = ndwi_full[row, col] > 0.0
        suspects.append(dict(id=fire["id"], area_ha=fire["area_ha"],
                             lon=c.x, lat=c.y, row=row, col=col,
                             flagged_as_water=bool(is_water_here)))

suspects_df = pd.DataFrame(suspects)
water_flagged = suspects_df[suspects_df["flagged_as_water"]]
print(f"{len(water_flagged)} / {len(suspects_df)} fires have centroids over water per our own NDWI mask")
display(water_flagged)

19 / 25 fires have centroids over water per our own NDWI mask


,id,area_ha,lon,lat,row,col,flagged_as_water
0,271189,158,13.386518,37.365569,2806,1813,True
1,271216,77,13.323336,37.415477,1775,3224,True
3,274690,527,13.421455,37.427439,1278,1472,True
4,276441,390,13.413994,37.362951,2806,1293,True
5,270168,17,13.299725,37.397468,2249,3571,True
6,270866,22,13.422385,37.341843,3283,1034,True
7,270867,22,13.370554,37.371561,2700,2137,True
8,271059,52,13.403170,37.395989,2056,1655,True
9,271060,11,13.385132,37.352681,3111,1775,True
10,271217,12,13.356581,37.395403,2172,2512,True


In [11]:
# Get direct Google Maps links for JUST the flagged ones
for _, row in water_flagged.iterrows():
    print(f"fire {row['id']} ({row['area_ha']:.0f} ha): "
          f"https://www.google.com/maps?q={row['lat']:.5f},{row['lon']:.5f}")

fire 271189 (158 ha): https://www.google.com/maps?q=37.36557,13.38652
fire 271216 (77 ha): https://www.google.com/maps?q=37.41548,13.32334
fire 274690 (527 ha): https://www.google.com/maps?q=37.42744,13.42145
fire 276441 (390 ha): https://www.google.com/maps?q=37.36295,13.41399
fire 270168 (17 ha): https://www.google.com/maps?q=37.39747,13.29973
fire 270866 (22 ha): https://www.google.com/maps?q=37.34184,13.42239
fire 270867 (22 ha): https://www.google.com/maps?q=37.37156,13.37055
fire 271059 (52 ha): https://www.google.com/maps?q=37.39599,13.40317
fire 271060 (11 ha): https://www.google.com/maps?q=37.35268,13.38513
fire 271217 (12 ha): https://www.google.com/maps?q=37.39540,13.35658
fire 271225 (13 ha): https://www.google.com/maps?q=37.40781,13.36965
fire 271236 (10 ha): https://www.google.com/maps?q=37.39647,13.34048
fire 271238 (26 ha): https://www.google.com/maps?q=37.31997,13.46728
fire 272728 (60 ha): https://www.google.com/maps?q=37.39763,13.37207
fire 272727 (12 ha): https://ww

In [12]:
true_colour = np.dstack([
    stretch(stack[IDX["RED"]], valid),
    stretch(stack[IDX["GREEN"]], valid),
    stretch(stack[IDX["BLUE"]], valid),
])

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(true_colour, extent=ext_geo, origin="upper", interpolation="nearest")
ax.set_xlim(ext_geo[0], ext_geo[1]); ax.set_ylim(ext_geo[2], ext_geo[3])
ax.set_title(f"Session {SESSION_ID}: TRUE COLOUR (R=665nm, G=560nm, B=490nm)")
fig.savefig(OUT_DIR / "true_colour_check.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print("saved true_colour_check.png")

saved true_colour_check.png


In [13]:
example = water_flagged.iloc[0]
r, c = example["row"], example["col"]
print(f"Fire {example['id']} at pixel (row={r}, col={c}), confirmed via Google Maps as LAND:")
for role, idx in IDX.items():
    print(f"  {role:<8}: {stack[idx, r, c]:.1f}")
print(f"  NDVI here: {ndvi_full[r, c]:.3f}")
print(f"  NDWI here: {ndwi_full[r, c]:.3f}  (positive = classified as water)")

Fire 271189 at pixel (row=2806, col=1813), confirmed via Google Maps as LAND:
  BLUE    : 9.0
  GREEN   : 14.0
  RED     : 12.0
  RE1     : 3.0
  RE2     : 3.0
  RE3     : 3.0
  NIR     : 3.0
  NDVI here: -0.600
  NDWI here: 0.647  (positive = classified as water)


In [14]:
print(f"{'band':<10}{'min':>6}{'max':>6}{'mean':>8}{'std':>8}{'unique values':>15}")
for role, idx in IDX.items():
    a = stack[idx][valid]
    n_unique = len(np.unique(a))
    print(f"{role:<10}{a.min():>6.0f}{a.max():>6.0f}{a.mean():>8.2f}{a.std():>8.2f}{n_unique:>15}")

band         min   max    mean     std  unique values
BLUE           5    44    8.61    1.99             40
GREEN          6    89   14.86    5.78             84
RED            7   206   24.84   15.31            166
RE1            1    73    8.47    6.27             63
RE2            1    89    9.99    8.30             72
RE3            1   128   13.80   11.87             94
NIR            1   101   11.27    9.34             71


In [15]:
# If you haven't already run this from earlier, do it now:
import rasterio

profile = {
    'driver': 'GTiff', 'height': H, 'width': W, 'count': 3, 'dtype': 'uint8',
    'crs': 'EPSG:4326', 'transform': fitted_transform,
}
out_path = str(OUT_DIR / "session2929_georeferenced_truecolour.tif")
tc_uint8 = (true_colour * 255).astype('uint8')
with rasterio.open(out_path, 'w', **profile) as dst:
    dst.write(tc_uint8[:, :, 0], 1)
    dst.write(tc_uint8[:, :, 1], 2)
    dst.write(tc_uint8[:, :, 2], 3)
print(f"saved: {out_path}")

saved: /root/projects/iride_onboard-burnscar-mapper/cgi_feedback/session2929_georeferenced_truecolour.tif


In [17]:
# Once you have both coordinate pairs from QGIS, compute the real distance:
from math import radians, sin, cos, sqrt, atan2

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000
    dlat, dlon = radians(lat2-lat1), radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

google_lat, google_lon = 37.4102, 13.3237
phisat_lat, phisat_lon = 37.4711, 13.3216

offset_m = haversine_m(google_lat, google_lon, phisat_lat, phisat_lon)
print(f"measured offset: {offset_m:.0f} metres")

measured offset: 6774 metres
